# Case study 3 on Kaggle

Use this notebook when the laptop does not have the memory for the 789 MB extract. Kaggle's CPU session has more RAM than an 8 GB machine, and the pipeline already reads the file in chunks, so a GPU is not required.

## Before you press Run

1. Zip the `case_study_3` folder. Leave `work/` out of the zip. That folder is a local cache and can be larger than the csv.
2. On Kaggle, create a dataset from that zip. Kaggle unpacks it under `/kaggle/input/<your-dataset>/`.
3. Create a second dataset from `labeled_power_source_data.csv` (789 MB). Keep the file name if you can.
4. New Notebook. Settings: **Internet On** (the first cell installs `sktime`, `numba`, and `shap`), **Accelerator None**. Add both datasets with *Add Data*.
5. Upload this notebook, or paste these cells, and Run All.

The first code cell finds the code and the csv, then points caches at `/kaggle/working`, which is the only writable directory. Training writes `artifacts/` there. Download that folder when the run finishes if you want the API on your laptop: copy it over `case_study_3/artifacts`.

A full local run, once the csv was on disk, finished in about four minutes. Expect a similar order of magnitude here, plus the SHAP step at the end.

## Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ronnie-Leon76/Kamilimu-2026-DS/blob/case-study-3-pipeline/Deploying%20ML%20Solutions/case_study_3/kaggle_power_source.ipynb)

Colab does not keep the Python package beside the notebook. The first code cell clones the same public repository, so `pipeline/` and `bootstrap.py` are in the runtime before training starts. Upload `labeled_power_source_data.csv` to Drive and set `POWER_SOURCE_CSV` in that cell. Colab's disk is large enough for this extract. A GPU is still unnecessary.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Colab opens this .ipynb alone. The clone brings pipeline/, service/, tests/, and artifacts/.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Ronnie-Leon76/Kamilimu-2026-DS.git"
BRANCH = "case-study-3-pipeline"
CLONE = Path("/content/Kamilimu-2026-DS")

if IN_COLAB:
    package = CLONE / "Deploying ML Solutions" / "case_study_3" / "pipeline" / "config.py"
    if not package.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(CLONE),
        ])
    os.chdir(CLONE)
    requirements = CLONE / "Deploying ML Solutions" / "case_study_3" / "requirements.txt"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
    print("repository:", CLONE)
    print("pipeline:", package.parent)
    print("artifacts:", CLONE / "Deploying ML Solutions" / "case_study_3" / "artifacts")
else:
    print("Not Colab. Using the pipeline folder already on this machine.")

# Optional, on Colab, after uploading the extract to Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# os.environ["POWER_SOURCE_CSV"] = "/content/drive/MyDrive/labeled_power_source_data.csv"

import os
import sys
from pathlib import Path

def find(name, roots):
    found = []
    for root in roots:
        if root.exists():
            found.extend(root.rglob(name))
    return found

roots = [Path("/kaggle/input"), Path("/content/drive/MyDrive"), Path("/content"), Path.cwd()]
bootstraps = find("bootstrap.py", roots)
if not bootstraps:
    raise FileNotFoundError(
        "The pipeline was not found. On Kaggle, add the case_study_3 code as a dataset. On Colab, rerun this cell so the clone finishes."
    )
case_dir = next(path.parent for path in bootstraps if (path.parent / "pipeline").is_dir())
sys.path.insert(0, str(case_dir))

from bootstrap import configure
info = configure()
for key, value in info.items():
    print(f"{key}: {value}")
if not info["csv"]:
    raise FileNotFoundError("labeled_power_source_data.csv was not found. On Kaggle, add it as a dataset. On Colab, set POWER_SOURCE_CSV to the file on Drive.")


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "sktime==1.2.0", "numba", "shap==0.53.0", "pyarrow",
])


## Train

`run()` partitions the extract, builds causal features, fits PCA, logistic, boosting, forest, temporal folds, the window models, the forecast, the figures, and then SHAP. A second run in the same session reuses `/kaggle/working/work`.

In [ ]:
from pipeline.train import run

metrics = run(force_rebuild=False)
print("deployed:", metrics["default_strategy"])
print("rows:", metrics["rows_in_feature_store"])


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

artifacts = Path(os.environ["ARTIFACT_DIR"])
figures = artifacts / "figures"
metrics = json.loads((artifacts / "metrics.json").read_text(encoding="utf-8"))
display(pd.DataFrame(metrics["comparison"]).sort_values("multi_macro_f1", ascending=False))
for name in (
    "model_comparison.png",
    "confusion_matrix.png",
    "shap_mean_abs.png",
    "shap_beeswarm_generator.png",
    "shap_dependence.png",
    "shap_waterfall.png",
    "frequency_forecast.png",
    "generator_hours.png",
):
    path = figures / name
    print(name if path.exists() else f"missing {name}")
    if path.exists():
        display(Image(filename=str(path)))